# TP1 : Découpage et vectorisation

**Contexte :** assistant du service client de Lagune Banque.

Un RAG ne lit pas des documents entiers : il en retrouve des chunks. Vous allez comparer deux méthodes de découpage, mesurer des distances entre embeddings, puis construire six index pour voir lesquels conservent les réponses complètes.

```mermaid
flowchart LR
    D["Documents : rag.load_corpus"] --> S["Splitter"]
    S --> C["Chunks"]
    C --> E["Embeddings : rag.embeddings"]
    E --> I["Index Weaviate : rag.index_chunks"]
```

Complétez les `# TODO`.

## <a id="syntaxe"></a>Syntaxe

Le splitter de LangChain découpe un texte en chunks d'au plus `chunk_size` caractères, avec `chunk_overlap` caractères d'overlap. `separators` donne les séparateurs par ordre de préférence :

```python
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=75, separators=["\n\n", "\n", ". ", " ", ""])
chunks = splitter.split_text(texte)   # liste de str
```

`rag.index_chunks(client, nom, chunks)` calcule l'embedding de chaque chunk et le range dans une collection. `rag.search(collection, question, k)` renvoie les `k` chunks les plus proches de la question : une liste de `{texte, distance, meta}`.

## <a id="tp1-configuration"></a>Configuration

In [ ]:
import re
import sys
import unicodedata
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(next(p for p in [Path("code_partage"), Path("../code_partage"), Path("../../code_partage")] if (p / "rag.py").exists()).resolve()))
import rag
from config import prepare
from weaviate_store import connect

In [ ]:
prepare(embedding=True)

## <a id="documents"></a>Documents

Le corpus compte dix documents dans trois formats (PDF, HTML, Word), décrits par un manifeste : identifiant `DOC-xx`, version, dates de validité. `rag.load_corpus()` renvoie le texte de chaque page avec ces métadonnées.

La note de service DOC-10 est un scan : aucun texte ne peut en être extrait. `rag.load_corpus` utilise à la place sa transcription OCR (`note_service_incident.ocr.txt`), fournie avec le corpus. Sans elle, le document serait invisible pour le RAG.

In [ ]:
documents = rag.load_corpus()
for d in documents:
    if d["fichier"].endswith(".pdf") and not d["avertissement"]:
        d["pages"] = rag.strip_headers(d["pages"])
pd.DataFrame([{"id": d["id"], "titre": d["titre"][:45], "pages": len(d["pages"]), "caractères": sum(len(t) for _, t in d["pages"])} for d in documents])

## <a id="pourquoi-decouper"></a>Pourquoi découper

Un chunk doit contenir une réponse complète. Prenons la règle du plafond du Prêt Express (conditions générales, article 4.2) : elle tient en 250 caractères environ.

In [ ]:
texte = next(t for _, t in documents[0]["pages"] if "4.2 Le plafond" in t)
debut = texte.index("4.2 Le plafond")
print(texte[debut:debut + 255])

Si un chunk coupe cette règle en deux, une question sur le plafond récupère une moitié de la réponse : le plafond de départ sans la façon dont il augmente, ou l'inverse. Le modèle de langage ne reçoit que les chunks retrouvés.

## <a id="splitter-fixe"></a>Splitter fixe

Le chunking le plus simple coupe le texte en chunks de `chunk_size` caractères. Chaque chunk commence `chunk_size - overlap` caractères après le début du précédent. L'overlap répète la fin d'un chunk au début du suivant : une phrase coupée à la frontière reste entière dans l'un des deux.

**TODO 1** : complétez la ligne qui fait avancer le début du chunk suivant, pour que deux chunks consécutifs partagent `overlap` caractères.

In [ ]:
def split_fixed(texte, chunk_size, overlap):
    chunks, debut = [], 0
    while True:
        chunks.append(texte[debut:debut + chunk_size])
        if debut + chunk_size >= len(texte):    # ce chunk atteint la fin du texte
            return chunks
        # TODO 1 : faites avancer le début du chunk suivant en gardant `overlap` caractères en commun
        debut += chunk_size

Test sur l'alphabet, avec dix lettres par chunk et deux d'overlap. Attendu : `['abcdefghij', 'ijklmnopqr', 'qrstuvwxyz']`.

In [ ]:
split_fixed("abcdefghijklmnopqrstuvwxyz", 10, 2)

Appliquez-le à l'article 4.2, avec des chunks de 200 caractères.

In [ ]:
extrait = texte[max(debut - 60, 0):debut + 600]
for i, chunk in enumerate(split_fixed(extrait, 200, 30)):
    print(f"[{i}] {chunk!r}")

Observez où les chunks sont coupés : au milieu d'un mot ou d'un chiffre, au milieu d'une phrase. Le splitter ne sait pas où s'arrête une idée.

## <a id="splitter-recursif"></a>Splitter récursif

Le splitter de LangChain (`RecursiveCharacterTextSplitter`, utilisé par `rag.split_documents`) essaie de couper d'abord aux séparateurs les plus marqués : saut de paragraphe, saut de ligne, fin de phrase, espace, et un caractère seulement en dernier recours. Il ne passe au séparateur suivant que si le chunk dépasse encore la taille maximale.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

recursif = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30, separators=["\n\n", "\n", ". ", " ", ""])
for i, chunk in enumerate(recursif.split_text(extrait)):
    print(f"[{i}] {chunk!r}")

Mêmes 200 caractères, même overlap de 30 : les chunks commencent et finissent plus souvent à une frontière de phrase.

## <a id="configurations"></a>Configurations

Deux splitters (le vôtre et le récursif) et trois tailles. L'overlap vaut environ 15 % de la taille.

| Taille | `chunk_size` | `overlap` |
|---|---|---|
| Petit | 200 | 30 |
| Moyen | 500 | 75 |
| Grand | 1000 | 150 |

`chunks_from` donne à votre splitter le même format de sortie que `rag.split_documents` : un dict par chunk, avec son texte et ses métadonnées (document, version, dates, page).

In [ ]:
def chunks_from(documents, splitter, nom):
    chunks = []
    for d in documents:
        for page, texte in d["pages"]:
            for i, chunk in enumerate(splitter(texte)):
                if chunk.strip():
                    chunks.append({"id": f"{nom}_{d['id']}_p{page}_{i}", "texte": chunk,
                                     "meta": {"source": d["id"], "titre": d["titre"], "version": d["version"],
                                              "date_effet": d["date_effet"] or "", "date_fin": d["date_fin"] or "", "page": str(page)}})
    return chunks


TAILLES = {"petit": (200, 30), "moyen": (500, 75), "grand": (1000, 150)}
CONFIGS = {}
for nom, (chunk_size, overlap) in TAILLES.items():
    CONFIGS[f"fixe_{nom}"] = chunks_from(documents, lambda t, a=chunk_size, b=overlap: split_fixed(t, a, b), f"fixe_{nom}")
    CONFIGS[f"recursif_{nom}"] = rag.split_documents(documents, chunk_size, overlap, f"recursif_{nom}")

Pour chaque configuration : le nombre de chunks, leur longueur moyenne, et la part de ceux qui se terminent par une ponctuation de fin de phrase.

In [ ]:
def ends_cleanly(chunk):
    return chunk.rstrip().endswith((".", "?", "!", ":", ";"))


pd.DataFrame([{"configuration": nom, "chunks": len(ps), "longueur moyenne": round(sum(len(p["texte"]) for p in ps) / max(len(ps), 1)),
               "finissent proprement": f"{100 * sum(ends_cleanly(p['texte']) for p in ps) / max(len(ps), 1):.0f} %"}
              for nom, ps in CONFIGS.items()])

## <a id="embeddings-distance"></a>Embeddings et distance

Un embedding transforme un texte en vecteur : deux textes de sens proche donnent deux vecteurs proches, même sans mots communs. C'est ce qui permet de retrouver un chunk à partir d'une question. Le modèle d'embedding (langue, dimension, coût) compte autant que le chunking. Ici, c'est `MODELE_EMBEDDING`, appelé via LiteLLM.

La base vectorielle compare les vecteurs avec la distance `l2-squared` : la somme des carrés des écarts entre coordonnées. Plus la valeur est petite, plus les textes sont proches.

**TODO 2** : dans `distance(v1, v2)`, complétez la ligne qui ajoute, pour chaque coordonnée, le carré de l'écart entre les deux vecteurs.

In [ ]:
def distance(v1, v2):
    total = 0
    for a, b in zip(v1, v2):    # a et b : la même coordonnée dans les deux vecteurs
        # TODO 2 : ajoutez le carré de l'écart entre a et b
        total += 0
    return total

Test : la distance entre `[0, 0]` et `[3, 4]` doit valoir 25.

In [ ]:
distance([0, 0], [3, 4])

Une question, un chunk qui y répond, un chunk hors sujet, et une paraphrase de la question qui n'a presque aucun mot en commun avec le chunk.

In [ ]:
question = "Quels sont les frais de retard du Prêt Express ?"
pertinent = "Le Prêt Express coûte 2 000 FCFA de pénalité forfaitaire quand une échéance reste impayée."
hors_sujet = "Les intérêts de l'Épargne Plus sont versés chaque trimestre sur le solde du compte."
paraphrase = "Que dois-je payer en plus si je rembourse après la date prévue ?"

vq, vp, vh, vpa = rag.embeddings([question, pertinent, hors_sujet, paraphrase])
print("dimension du vecteur :", len(vq))
print(f"question   -> chunk pertinent  : {distance(vq, vp):.3f}")
print(f"question   -> chunk hors sujet : {distance(vq, vh):.3f}")
print(f"paraphrase -> chunk pertinent  : {distance(vpa, vp):.3f}")
print(f"paraphrase -> chunk hors sujet : {distance(vpa, vh):.3f}")

## <a id="index"></a>Index

Une collection vectorielle par configuration : `index_fixe_petit`, `index_recursif_moyen`, etc. `rag.index_chunks` calcule l'embedding de chaque chunk et le stocke avec son texte et ses métadonnées.

**TODO 3** : pour chaque configuration de `CONFIGS`, indexez ses chunks dans une collection nommée `f"index_{nom}"` avec `rag.index_chunks(client, nom_collection, chunks)`, et rangez-la dans `index_par_config[nom]`.

In [ ]:
client = connect()
index_par_config = {}
# TODO 3

## <a id="evaluation-chunks"></a>Évaluation des chunks

La bonne question n'est pas « le bon document est-il renvoyé ? » mais « le chunk renvoyé contient-il la réponse entière ? ». Chaque question ci-dessous est décrite par les éléments qui doivent figurer ensemble dans un même chunk parmi les trois premiers. Une réponse coupée en deux chunks compte comme un échec.

In [ ]:
QUESTIONS = [
    ("Quels sont les frais de retard du Prêt Express+ ?", ["prêt express+", "5 000 fcfa", "échéance impayée"]),
    ("Quels sont les frais de retard du Prêt Express ?", ["prêt express", "2 000 fcfa", "forfait"]),
    ("Quel est le plafond du Prêt Express après 6 prêts remboursés ?", ["15 000", "série de 2", "50 000"]),
    ("Quand les intérêts de l'Épargne Plus sont-ils versés ?", ["trimestriellement", "31/12"]),
    ("Le service client est-il joignable le dimanche ?", ["lundi au samedi", "8 h"]),
    ("Combien coûte un Prêt Express+ sur 60 jours ?", ["6 %", "3 % par tranche de 30 jours"]),
    ("Quel est le montant d'un Prêt Express ?", ["5 000 fcfa", "50 000 fcfa"]),
]


def normalize(texte):
    return re.sub(r"\s+", " ", unicodedata.normalize("NFKC", texte)).casefold()


def answer_complete(chunks, elements):
    return any(all(e in normalize(p["texte"]) for e in elements) for p in chunks)


resultats = {nom: [answer_complete(rag.search(index, question, 3), elements) for question, elements in QUESTIONS]
             for nom, index in index_par_config.items()}
tableau = pd.DataFrame({nom: ["✓" if ok else "✗" for ok in oks] for nom, oks in resultats.items()}, index=[q[:48] for q, _ in QUESTIONS]).T
tableau["réponses complètes"] = [f"{sum(oks)} / {len(QUESTIONS)}" for oks in resultats.values()]
tableau

Lisez un échec : voici ce que renvoie la première configuration qui rate une question.

In [ ]:
echec = next(((nom, i) for nom, oks in resultats.items() for i, ok in enumerate(oks) if not ok), None)
if echec:
    nom, i = echec
    question, elements = QUESTIONS[i]
    print(f"Configuration {nom} | question : {question} | éléments attendus ensemble : {elements}\n")
    for rang, p in enumerate(rag.search(index_par_config[nom], question, 3), 1):
        print(f"[{rang}] {p['meta']['source']} | {p['texte']!r}")

## Contrôle automatique

La dernière cellule affiche ÉCHEC tant que les `# TODO` ne sont pas faits.

In [ ]:
try:
    assert split_fixed("abcdefghijklmnopqrstuvwxyz", 10, 2) == ["abcdefghij", "ijklmnopqr", "qrstuvwxyz"], "split_fixed : voir l'exemple de l'alphabet (TODO 1)"
    assert all(len(p) <= 200 for p in split_fixed(texte, 200, 30)), "split_fixed : un chunk dépasse la taille (TODO 1)"
    assert distance([0, 0], [3, 4]) == 25, "distance : somme des carrés des écarts (TODO 2)"
    assert distance(vq, vp) < distance(vq, vh), "le chunk pertinent doit être plus proche de la question que le chunk hors sujet"
    assert set(index_par_config) == set(CONFIGS), "il faut un index par configuration (TODO 3)"
    assert rag.search(index_par_config["recursif_moyen"], "Quels sont les frais de retard du Prêt Express ?", 2), "l'index recursif_moyen ne renvoie rien (TODO 3)"
    print("OK")
except Exception as e:
    print(f"ÉCHEC : {e if isinstance(e, AssertionError) else 'terminez les # TODO et relancez les cellules.'}")